# CS-781 Phase 2 - Legal NER on our Phase-1 models
Accelerator **GPU T4 x2**, Internet On, HF secret ticked -> **Save Version -> Save & Run All**.

Finished runs (found on Hugging Face) are skipped automatically. Cell 7: round 1 + round 2 + round 3 (full-data twins of the best runs). Cell 8: results table. Cell 9: ensemble. Zips in `/kaggle/working/zips/`.


In [21]:
# ===================== P2 CELL 1 - Setup + download Phase-2 data, our tokenizer, the 4 Phase-1 models =====================
# Settings: Accelerator = GPU T4 x2, Internet = On, Secret "HF" ticked.
import os, sys, json, time, math, random, re, shutil, collections
os.environ["HF_HOME"] = "/tmp/hf_home"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download
_sec = UserSecretsClient()
try:
    HF_TOKEN = _sec.get_secret("HF")
except Exception:
    HF_TOKEN = _sec.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"          # our tokenizer lives here
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"          # Phase-1 models; Phase-2 outputs go to its phase2/ folder
SRC_REPO  = "Exploration-Lab/CS-781-Capstone"
WORK      = "/kaggle/working"
P2        = "/tmp/cs781/p2"; os.makedirs(P2, exist_ok=True)

# ---- official Phase-2 files ----
top = [x.path for x in api.list_repo_tree(SRC_REPO, repo_type="dataset")]
P2_DIR = next(p for p in top if p.lower().startswith("phase_2"))
p2_files = [x.path for x in api.list_repo_tree(SRC_REPO, repo_type="dataset", path_in_repo=P2_DIR, recursive=True)
            if getattr(x, "size", None) is not None]
print("Phase-2 files:", *p2_files, sep="\n   ")
LOCAL = {}
for p in p2_files:
    LOCAL[p.split("/")[-1]] = hf_hub_download(SRC_REPO, p, repo_type="dataset", token=HF_TOKEN, local_dir=f"{P2}/raw")
need = ["fold1-text.json", "fold1-gold.json", "fold2-text.json", "fold2-gold.json", "labels.txt", "fold3-text.json"]
missing = [n for n in need if n not in LOCAL]
assert not missing, f"missing files {missing} - paste this output to Claude"

# ---- our tokenizer (Phase 1) ----
TOK_PATH = hf_hub_download(DATA_REPO, "tokenizer.json", repo_type="dataset", token=HF_TOKEN, local_dir=f"{P2}/tok")
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOK_PATH)
print("tokenizer vocab:", tok.get_vocab_size())

# ---- which Phase-1 checkpoints exist ----
runs = sorted({f.split("/")[1] for f in api.list_repo_files(CKPT_REPO) if f.startswith("runs/")})
print("Phase-1 runs on HF:", runs)

import torch
print(f"torch {torch.__version__} | GPUs {torch.cuda.device_count()}",
      *[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Phase-2 files:
   Phase_2_LNER/Test/fold3-text.json
   Phase_2_LNER/Train/fold1-gold.json
   Phase_2_LNER/Train/fold1-text.json
   Phase_2_LNER/Train/fold2-gold.json
   Phase_2_LNER/Train/fold2-text.json
   Phase_2_LNER/Train/labels.txt
tokenizer vocab: 32768
Phase-1 runs on HF: ['dapt', 'dapt_llama', 'explore_encdec', 'explore_gpt2', 'explore_llama', 'explore_shared', 'main_llama', 'smoke_llama']
torch 2.11.0+cu128 | GPUs 2 Tesla T4 Tesla T4


In [22]:
%%writefile /kaggle/working/p2text.py
"""CS-781 Phase 2 (L-NER) text utilities: loading, normalisation WITH an exact character map, token alignment,
BIO encoding/decoding. Every offset that leaves this module is an offset into the ORIGINAL document text."""
import bisect, json, unicodedata

# same punctuation map as Phase-1 pre-processing (the models were pretrained on text normalised this way)
PUNCT = {"\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "\u2032": "'",
         "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"',
         "\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2013": "-", "\u2014": "-",
         "\u2015": "-", "\u2212": "-", "\u2022": "*", "\u00b7": "."}


def load_texts(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, list):                                   # tolerate [{"id":..,"text":..}, ...]
        d = {str(x.get("id")): x.get("text") for x in d}
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            v = v.get("text", "")
        out[str(k)] = v if isinstance(v, str) else ""
    return out


def _ent(x):
    if isinstance(x, dict):
        s = x.get("start", x.get("start_char", x.get("begin")))
        e = x.get("end", x.get("end_char"))
        l = x.get("label", x.get("type", x.get("entity")))
        return int(s), int(e), str(l)
    return int(x[0]), int(x[1]), str(x[2])


def load_gold(path):
    d = json.load(open(path, encoding="utf-8"))
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            v = v.get("entities", v.get("spans", []))
        out[str(k)] = [_ent(x) for x in (v or [])]
    return out


def normalize_with_map(text):
    """-> (norm, omap): norm is single-spaced ASCII (like Phase-1 training text); omap[k] = index in `text` of norm[k].
    Whitespace runs become one space (mapped to the first whitespace char); non-ASCII chars are NFKC-folded or dropped."""
    out, omap, prev_space = [], [], True
    for i, ch in enumerate(text):
        c = PUNCT.get(ch, ch)
        if c.isspace():
            pass
        elif ord(c) < 32 or ord(c) == 127 or unicodedata.category(c) == "Cc":   # invisible control chars: drop
            continue
        if c.isspace():
            if not prev_space:
                out.append(" "); omap.append(i); prev_space = True
            continue
        if ord(c) > 127:
            c = "".join(x for x in unicodedata.normalize("NFKC", c) if ord(x) < 128 and not x.isspace())
            if not c:                                   # accented letters: strip the accent (e-acute -> e)
                c = "".join(x for x in unicodedata.normalize("NFKD", ch) if ord(x) < 128 and not x.isspace())
            if not c:                                   # anything else (Hindi, symbols) is dropped
                continue
        for x in c:
            out.append(x); omap.append(i)
        prev_space = False
    if out and out[-1] == " ":
        out.pop(); omap.pop()
    return "".join(out), omap


def orig_to_norm_span(omap, s, e):
    """Original [s,e) -> normalised [a,b) covering the same characters (None if nothing survives)."""
    a = bisect.bisect_left(omap, s)
    b = bisect.bisect_left(omap, e)
    return (a, b) if b > a else None


def norm_to_orig_span(omap, a, b, text=None):
    """Normalised [a,b) -> original [s,e). With `text`, the span also absorbs characters that normalisation DROPPED
    when they touch it without whitespace (e.g. a Hindi letter or a rare symbol at the end of a word)."""
    s, e = omap[a], omap[b - 1] + 1
    if text is not None:
        while e < len(text) and not text[e].isspace() and (b >= len(omap) or omap[b] != e):
            e += 1
        while s > 0 and not text[s - 1].isspace() and (a == 0 or omap[a - 1] != s - 1):
            s -= 1
    return s, e


def _is_punct(c):
    return not (c.isalnum() or c.isspace())


def tokenize(tok, norm, split_punct=True):
    """-> ids, spans: spans[i] = (a,b) normalised chars of token i WITHOUT its leading space ((a,a) if pure space).
    split_punct: BPE merges runs of punctuation into one token (e.g. ").", ".[", "),"); an entity boundary can fall
    inside such a run, so every multi-character punctuation token is split into one token per character
    (all single characters are in the byte-level vocabulary, so ids stay valid)."""
    enc = tok.encode(norm, add_special_tokens=False)
    ids, spans = [], []
    for tid, (a, b) in zip(enc.ids, enc.offsets):
        lead = a < b and norm[a] == " "
        core_a = a + 1 if lead else a
        core = norm[core_a:b]
        if split_punct and len(core) > 1 and all(_is_punct(c) for c in core):
            new_ids, new_spans = [], []
            for j, c in enumerate(core):
                piece = ("\u0120" + c) if (j == 0 and lead) else c        # \u0120 = byte-level marker for a space
                pid = tok.token_to_id(piece)
                if pid is None and j == 0 and lead:                     # "<space>X" not in vocab: emit space + X
                    sp = tok.token_to_id("\u0120")
                    if sp is not None:
                        new_ids.append(sp); new_spans.append((core_a, core_a))
                    pid = tok.token_to_id(c)
                new_ids.append(pid); new_spans.append((core_a + j, core_a + j + 1))
            if all(x is not None for x in new_ids):
                ids += new_ids; spans += new_spans
                continue                                                # else: keep the original token unsplit
        ids.append(tid); spans.append((core_a, b))
    assert all(x is not None for x in ids), "tokenizer produced an unknown id"
    return ids, spans


def spans_to_bio(tok_spans, ents_norm, label2id):
    """ents_norm: list of (a,b,label) in normalised chars. Returns tag ids: 0=O, 2k+1=B-label_k, 2k+2=I-label_k.
    A token is inside an entity if its (trimmed) chars overlap it. Overlapping entities: the earlier/longer one wins."""
    tags = [0] * len(tok_spans)
    starts = [a for a, _ in tok_spans]
    for a, b, lab in sorted(ents_norm, key=lambda x: (x[0], -(x[1] - x[0]))):
        k = label2id[lab]
        i = max(0, bisect.bisect_right(starts, a) - 1)
        first = True
        while i < len(tok_spans) and tok_spans[i][0] < b:
            ta, tb = tok_spans[i]
            if tb > ta and tb > a and ta < b and tags[i] == 0:
                tags[i] = 2 * k + 1 if first else 2 * k + 2
                first = False
            i += 1
    for i in range(1, len(tags) - 1):          # a pure-space token inside an entity continues it
        if tok_spans[i][1] <= tok_spans[i][0] and tags[i - 1] and tags[i + 1] and tags[i + 1] % 2 == 0 \
                and (tags[i + 1] - 1) // 2 == (tags[i - 1] - 1) // 2:
            tags[i] = tags[i + 1]
    return tags


def bio_to_spans(tags, tok_spans, id2label):
    """Greedy BIO decoding (an I- without a matching open entity starts a new one). -> [(a,b,label)] normalised."""
    out, cur = [], None
    for t, (a, b) in zip(tags, tok_spans):
        if b <= a:                       # pure-space token: does not close or extend anything
            continue
        if t == 0:
            if cur: out.append(cur); cur = None
            continue
        lab = id2label[(t - 1) // 2]
        is_b = (t - 1) % 2 == 0
        if is_b or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: out.append(cur)
    return [tuple(x) for x in out]


def doc_roundtrip(tok, text, gold, label2id, id2label):
    """Gold -> tokens -> BIO -> spans -> original offsets. Returns (recovered set, gold set, n_tokens)."""
    norm, omap = normalize_with_map(text)
    ids, tsp = tokenize(tok, norm)
    ents = []
    for s, e, l in gold:
        r = orig_to_norm_span(omap, s, e)
        if r: ents.append((r[0], r[1], l))
    tags = spans_to_bio(tsp, ents, label2id)
    rec = {(*norm_to_orig_span(omap, a, b, text), l) for a, b, l in bio_to_spans(tags, tsp, id2label)}
    return rec, {(s, e, l) for s, e, l in gold}, len(ids)


def strict_macro(truth, pred, labels):
    """Same computation as evaluate_phase2.py. truth/pred: {doc: iterable of (s,e,label)}."""
    st = {l: [0, 0, 0] for l in labels}
    for d in truth:
        g = set(map(tuple, truth[d])); p = set(map(tuple, pred.get(d, [])))
        for l in labels:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            st[l][0] += len(gs & ps); st[l][1] += len(ps - gs); st[l][2] += len(gs - ps)
    per = {}
    for l in labels:
        tp, fp, fn = st[l]
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = (pr, rc, 2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    n = len(labels)
    return {"f1": sum(v[2] for v in per.values()) / n, "p": sum(v[0] for v in per.values()) / n,
            "r": sum(v[1] for v in per.values()) / n, "per_label": per}


Overwriting /kaggle/working/p2text.py


In [23]:
# ===================== P2 CELL 3 - Look at the data + alignment test + make our validation split =====================
sys.path.insert(0, WORK)
import importlib, p2text; importlib.reload(p2text)
from p2text import *
import numpy as np

LABELS = [x.strip() for x in open(LOCAL["labels.txt"], encoding="utf-8") if x.strip()]
L2I = {l: i for i, l in enumerate(LABELS)}; I2L = {i: l for l, i in L2I.items()}
print("labels:", LABELS)

texts = {f: load_texts(LOCAL[f"{f}-text.json"]) for f in ["fold1", "fold2", "fold3"]}
gold  = {f: load_gold(LOCAL[f"{f}-gold.json"]) for f in ["fold1", "fold2"]}
for f in texts:
    print(f"{f}: {len(texts[f])} docs" + (f", {sum(len(v) for v in gold[f].values())} entities, "
          f"ids equal to text ids: {set(gold[f]) == set(texts[f])}" if f in gold else " (test)"))

# ---- 1. raw format of one document and its first annotations ----
raw_gold = json.load(open(LOCAL["fold1-gold.json"], encoding="utf-8"))
d0 = next(iter(raw_gold))
print("\nRAW gold entry type:", type(raw_gold[d0]).__name__, "| first 3:", raw_gold[d0][:3] if isinstance(raw_gold[d0], list) else raw_gold[d0])
print("RAW text start:", repr(texts["fold1"][d0][:600]))

# ---- 2. text statistics ----
for f in texts:
    ch = [len(t) for t in texts[f].values()]
    nl = sum(t.count("\n") for t in texts[f].values()); na = sum(sum(ord(c) > 127 for c in t) for t in texts[f].values())
    print(f"{f}: chars/doc mean {np.mean(ch):,.0f} max {max(ch):,} | newlines {nl:,} | non-ASCII chars {na:,}")

# ---- 3. gold sanity: offsets in range, entity_text matches, overlaps, whitespace/punctuation at the edges ----
bad = mism = overl = edge_ws = 0; edge_examples = []
for f in gold:
    for d, ents in gold[f].items():
        t = texts[f][d]; ents_sorted = sorted(ents)
        for i, (s, e, l) in enumerate(ents_sorted):
            if not (0 <= s < e <= len(t)): bad += 1; continue
            if i and s < ents_sorted[i - 1][1]: overl += 1
            if t[s].isspace() or t[e - 1].isspace(): edge_ws += 1
        for x in (raw_gold.get(d, []) if f == "fold1" else []):
            if isinstance(x, list) and len(x) == 4 and texts[f][d][x[0]:x[1]] != x[3]: mism += 1
print(f"\nout-of-range spans {bad} | entity_text mismatches (fold1) {mism} | overlapping spans {overl} | spans with whitespace at an edge {edge_ws}")

# ---- 4. label counts ----
cnt = {f: collections.Counter(l for v in gold[f].values() for _, _, l in v) for f in gold}
print(f"\n{'label':10s} {'fold1':>7s} {'fold2':>7s}  example")
for l in LABELS:
    ex = next((texts[f][d][s:e] for f in gold for d, v in gold[f].items() for s, e, ll in v if ll == l), "")
    print(f"{l:10s} {cnt['fold1'][l]:7d} {cnt['fold2'][l]:7d}  {ex[:70]!r}")

# ---- 5. token lengths with OUR tokenizer (on the normalised text the model will see) ----
ntok = {f: [len(tokenize(tok, normalize_with_map(t)[0])[0]) for t in texts[f].values()] for f in texts}
for f in ntok:
    print(f"{f}: tokens/doc mean {np.mean(ntok[f]):,.0f}  median {np.median(ntok[f]):,.0f}  max {max(ntok[f]):,}  total {sum(ntok[f]):,}")

# ---- 6. ALIGNMENT TEST: gold -> tokens -> BIO -> back to original offsets, must be ~100% ----
rt_truth, rt_pred, miss_ex = {}, {}, []
for f in gold:
    for d, ents in gold[f].items():
        rec, g, _ = doc_roundtrip(tok, texts[f][d], ents, L2I, I2L)
        rt_truth[d], rt_pred[d] = g, rec
        for s, e, l in sorted(g - rec)[:2]:
            if len(miss_ex) < 15: miss_ex.append((l, texts[f][d][max(0, s - 15):s] + "[[" + texts[f][d][s:e] + "]]" + texts[f][d][e:e + 15]))
rt = strict_macro(rt_truth, rt_pred, LABELS)
print(f"\nALIGNMENT ROUND-TRIP strict macro-F1 = {rt['f1']:.4f}  (P {rt['p']:.4f} R {rt['r']:.4f})")
print("  per label F1:", {l: round(v[2], 4) for l, v in rt["per_label"].items()})
for l, s in miss_ex: print("  lost:", l, repr(s))

# ---- 7. do repeated mentions share a label? (tells us whether document-level label propagation can help) ----
prop = collections.defaultdict(lambda: [0, 0])          # label -> [occurrences of the string that are gold w/ same label, all occurrences]
for f in gold:
    for d, ents in gold[f].items():
        t = texts[f][d]; gs = {(s, e): l for s, e, l in ents}
        for surf, l in {(t[s:e], l) for s, e, l in ents if e - s >= 4}:
            for m in re.finditer(re.escape(surf), t):
                prop[l][1] += 1; prop[l][0] += gs.get((m.start(), m.end())) == l
print("\nRepeated surface strings that are gold with the SAME label (share):")
print("  ", {l: f"{a / b:.2f} of {b}" for l, (a, b) in sorted(prop.items())})

# ---- 8. our validation split: 15% of documents from each training fold (fixed seed) ----
rng = random.Random(781)
VAL_IDS = []
for f in ["fold1", "fold2"]:
    ids = sorted(gold[f]); rng.shuffle(ids); VAL_IDS += ids[: max(1, round(0.15 * len(ids)))]
ALL_TEXT = {**texts["fold1"], **texts["fold2"]}; ALL_GOLD = {**gold["fold1"], **gold["fold2"]}
TRAIN_IDS = [d for d in sorted(ALL_GOLD) if d not in set(VAL_IDS)]
split = {"train": TRAIN_IDS, "val": sorted(VAL_IDS), "test": sorted(texts["fold3"])}
os.makedirs(f"{P2}/data", exist_ok=True)
json.dump(split, open(f"{P2}/data/split.json", "w"), indent=1)
json.dump({d: [list(x) for x in ALL_GOLD[d]] for d in split["val"]}, open(f"{P2}/data/val_truth.json", "w"))
shutil.copy(LOCAL["labels.txt"], f"{P2}/data/labels.txt")
vc = collections.Counter(l for d in split["val"] for _, _, l in ALL_GOLD[d])
print(f"\nsplit: train {len(TRAIN_IDS)} docs | val {len(VAL_IDS)} docs | test {len(split['test'])} docs")
print("val entities per label:", dict(vc))
for fn in ["split.json", "val_truth.json", "labels.txt"]:
    api.upload_file(path_or_fileobj=f"{P2}/data/{fn}", path_in_repo=f"phase2/data/{fn}", repo_id=CKPT_REPO, repo_type="model")
print("uploaded split + val truth to", f"{CKPT_REPO}/phase2/data/")


labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
fold1: 35 docs, 4120 entities, ids equal to text ids: True
fold2: 35 docs, 4273 entities, ids equal to text ids: True
fold3: 35 docs (test)

RAW gold entry type: list | first 3: [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana']]
RAW text start: 'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the

No files have been modified since last commit. Skipping to prevent empty commit.


uploaded split + val truth to Abhayk24/cs781-p1-ckpt/phase2/data/


In [24]:
# ===================== P2 CELL 4 - Re-check alignment with punctuation splitting + a BALANCED validation split =====================
import importlib, p2text; importlib.reload(p2text)
from p2text import *

# (a) alignment round-trip again, now that punctuation runs such as ").", ".[" are split into single tokens
rt_truth, rt_pred = {}, {}
for f in gold:
    for d, ents in gold[f].items():
        rec, g, _ = doc_roundtrip(tok, texts[f][d], ents, L2I, I2L)
        rt_truth[d], rt_pred[d] = g, rec
rt = strict_macro(rt_truth, rt_pred, LABELS)
print(f"ALIGNMENT ROUND-TRIP (punctuation split) strict macro-F1 = {rt['f1']:.4f}")
print("  per label:", {l: round(v[2], 4) for l, v in rt["per_label"].items()})

# (b) the first split had only 3 WIT entities in validation -> a noisy macro-F1. Search many random 5+5-document
#     splits and keep the one whose WORST label still has the largest share of its entities in validation.
total = collections.Counter(l for v in ALL_GOLD.values() for _, _, l in v)
f1_ids, f2_ids = sorted(gold["fold1"]), sorted(gold["fold2"])
best = None
rng = random.Random(781)
for trial in range(20000):
    v = rng.sample(f1_ids, 5) + rng.sample(f2_ids, 5)
    c = collections.Counter(l for d in v for _, _, l in ALL_GOLD[d])
    share = [c[l] / total[l] for l in LABELS]
    key = (min(share), -abs(sum(c.values()) / sum(total.values()) - 0.15))
    if best is None or key > best[0]:
        best = (key, sorted(v), c)
VAL_IDS = best[1]
TRAIN_IDS = [d for d in sorted(ALL_GOLD) if d not in set(VAL_IDS)]
split = {"train": TRAIN_IDS, "val": VAL_IDS, "test": sorted(texts["fold3"])}
json.dump(split, open(f"{P2}/data/split.json", "w"), indent=1)
json.dump({d: [list(x) for x in ALL_GOLD[d]] for d in VAL_IDS}, open(f"{P2}/data/val_truth.json", "w"))
print(f"\nbalanced split: train {len(TRAIN_IDS)} | val {len(VAL_IDS)} | test {len(split['test'])}")
print(f"{'label':10s} {'val':>5s} {'total':>6s} {'share':>6s}")
for l in LABELS:
    print(f"{l:10s} {best[2][l]:5d} {total[l]:6d} {best[2][l] / total[l]:6.1%}")
for fn in ["split.json", "val_truth.json", "labels.txt"]:
    api.upload_file(path_or_fileobj=f"{P2}/data/{fn}", path_in_repo=f"phase2/data/{fn}", repo_id=CKPT_REPO, repo_type="model")
print("uploaded the balanced split to", f"{CKPT_REPO}/phase2/data/")


ALIGNMENT ROUND-TRIP (punctuation split) strict macro-F1 = 0.9943
  per label: {'APP': 0.9828, 'RESP': 0.9948, 'A.COUNSEL': 1.0, 'R.COUNSEL': 1.0, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.991, 'COURT': 0.9822, 'STAT': 0.9857, 'PREC': 0.9949, 'DATE': 1.0, 'CASENO': 1.0}

balanced split: train 60 | val 10 | test 35
label        val  total  share
APP           80    407  19.7%
RESP         148    381  38.8%
A.COUNSEL     35    195  17.9%
R.COUNSEL     35    184  19.0%
JUDGE         54    245  22.0%
WIT           51    187  27.3%
AUTH         410   1003  40.9%
COURT        178   1013  17.6%
STAT         366   1260  29.0%
PREC         306   1183  25.9%
DATE         302   1575  19.2%
CASENO       154    760  20.3%


No files have been modified since last commit. Skipping to prevent empty commit.


uploaded the balanced split to Abhayk24/cs781-p1-ckpt/phase2/data/


In [25]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)


Overwriting /kaggle/working/model.py


In [26]:
%%writefile /kaggle/working/p2ner.py
"""CS-781 Phase 2: Legal NER on top of our own Phase-1 language models.

One script for every experiment:
  --method full   : full fine-tuning (backbone + head)
  --method head   : frozen backbone, only the tagging head is trained (linear probe)
  --method lora   : LoRA adapters  W x + s*B(A x)            (own implementation)
  --method lorma  : LoRMA adapters (I + s*B A) W x  = y + s*B(A y)   (own implementation, additive-identity variant)
  --method bitfit : only bias / norm parameters + head
  --kd_teacher R  : knowledge distillation from Phase-2 run R (its best.pt) into this (student) model
Tagging: BIO over 12 labels (25 tags), constrained Viterbi decoding, optional document-level label propagation.
All offsets are exact character offsets of the original documents (p2text.py).
"""
import argparse, json, math, os, random, re, sys, time, zipfile
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig                      # our Phase-1 model code (unchanged)
import p2text as PT


# ================================================================================================================
# parameter-efficient adapters (written from scratch)
# ================================================================================================================
class LoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * B(A(dropout(x)))   ;  A ~ N(0, 1/r), B = 0  ->  starts exactly at W0."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel & Modi, 2025), additive rank inflation:  W' = (I + (alpha/r) * B A) W0.
    Computed without forming d x d matrices:  y0 = W0 x ;  y = y0 + s * B(A y0) (+ bias).
    A ~ N(0, 1/d_out), B = 0  ->  W' = W0 at the start (identity transform)."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        d_out = base.out_features
        self.A = nn.Parameter(torch.randn(r, d_out) / math.sqrt(d_out))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


def add_adapters(root: nn.Module, kind, targets, r, alpha, dropout):
    """Replace every nn.Linear whose attribute name is in `targets` (e.g. q,k,v,o) inside `root`."""
    cls = LoRALinear if kind == "lora" else LoRMALinear
    n = 0
    for mod in list(root.modules()):
        for name, child in list(mod.named_children()):
            if name in targets and isinstance(child, nn.Linear):
                setattr(mod, name, cls(child, r, alpha, dropout)); n += 1
    return n


# ================================================================================================================
# the NER network: Phase-1 backbone + tagging head
# ================================================================================================================
class NERNet(nn.Module):
    def __init__(self, lm: LegalLM, n_tags, bidir=True, head="linear", dropout=0.1):
        super().__init__()
        self.lm, self.arch, self.bidir = lm, lm.cfg.arch, bidir
        d = lm.cfg.d_model
        if self.arch == "encdec":                      # NER uses only the (bidirectional) encoder
            lm.blocks = nn.ModuleList()                # drop the decoder: not used, saves memory and parameters
        self.drop = nn.Dropout(dropout)
        self.head_kind = head
        if head == "bilstm":
            self.rnn = nn.LSTM(d, 256, batch_first=True, bidirectional=True)
            self.out = nn.Linear(512, n_tags)
        else:
            self.out = nn.Linear(d, n_tags)

    def features(self, idx):
        lm = self.lm
        if self.arch == "encdec":
            return lm.encode(idx)
        x = lm._embed(idx)
        for blk in lm._decoder_layers():
            x = blk(x, rope=lm._rope(), pos=0, causal=not self.bidir)
        return lm.norm(x)

    def forward(self, idx):
        h = self.drop(self.features(idx))
        if self.head_kind == "bilstm":
            h, _ = self.rnn(h.float())
        return self.out(h).float()


def count_params(m):
    tot = sum(p.numel() for p in m.parameters())
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tot, tr


# ================================================================================================================
# data
# ================================================================================================================
class Doc:
    def __init__(self, did, text, tok, gold=None, L2I=None):
        self.id, self.text = did, text
        self.norm, self.omap = PT.normalize_with_map(text)
        self.ids, self.spans = PT.tokenize(tok, self.norm)
        self.gold = gold
        self.tags = None
        if gold is not None:
            ents = []
            for s, e, l in gold:
                r = PT.orig_to_norm_span(self.omap, s, e)
                if r: ents.append((r[0], r[1], l))
            self.tags = PT.spans_to_bio(self.spans, ents, L2I)


def windows(n, W, stride):
    if n == 0:
        return []
    if n <= W:
        return [(0, n)]
    starts = list(range(0, n - W + 1, stride))
    if starts[-1] != n - W:
        starts.append(n - W)
    return [(s, s + W) for s in starts]


def train_batches(docs, W, bs, rng):
    """Every epoch: each document is cut into length-W windows from a random offset (+ a final window);
    windows are grouped by length so no padding is ever needed."""
    items = []
    for di, d in enumerate(docs):
        n = len(d.ids)
        if n < 2:                                         # empty / one-token document: nothing to learn from
            continue
        if n <= W:
            items.append((di, 0, n)); continue
        off = rng.randrange(0, W // 2)
        s = off
        while s + W <= n:
            items.append((di, s, s + W)); s += W
        items.append((di, n - W, n))
        if off > 0:
            items.append((di, 0, W))
    rng.shuffle(items)
    by_len = defaultdict(list)
    for it in items:
        by_len[it[2] - it[1]].append(it)
    batches = []
    for L, its in by_len.items():
        for i in range(0, len(its), bs):
            batches.append(its[i:i + bs])
    rng.shuffle(batches)
    return batches


# ================================================================================================================
# decoding
# ================================================================================================================
def allowed_transitions(n_labels):
    """BIO constraints: I-X may only follow B-X or I-X. Tag ids: 0=O, 2k+1=B-k, 2k+2=I-k."""
    T = 2 * n_labels + 1
    ok = np.ones((T, T), dtype=bool)                      # ok[prev, cur]
    start_ok = np.ones(T, dtype=bool)
    for cur in range(1, T):
        if cur % 2 == 0:                                  # I-k
            k = (cur - 1) // 2
            start_ok[cur] = False
            for prev in range(T):
                if prev not in (2 * k + 1, 2 * k + 2):
                    ok[prev, cur] = False
    return ok, start_ok


def viterbi(logp, ok, start_ok):
    """logp: (n, T) log-probabilities -> best tag sequence obeying BIO constraints."""
    n, T = logp.shape
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    trans = np.where(ok, 0.0, -1e9)
    score = logp[0] + np.where(start_ok, 0.0, -1e9)
    back = np.zeros((n, T), dtype=np.int32)
    for i in range(1, n):
        cand = score[:, None] + trans                     # (prev, cur)
        back[i] = cand.argmax(0)
        score = cand.max(0) + logp[i]
    tags = np.zeros(n, dtype=np.int64)
    tags[-1] = int(score.argmax())
    for i in range(n - 1, 0, -1):
        tags[i - 1] = back[i, tags[i]]
    return tags


@torch.no_grad()
def predict_logprobs(net, doc, W, stride, device, amp, bs=8):
    """Sliding windows; each token's probabilities are averaged over the windows that contain it, weighted by how
    central the token is in each window (edges of a window see little context)."""
    n = len(doc.ids)
    if n == 0:                                            # empty document -> no tokens, no entities
        return np.zeros((0, net.out.out_features), dtype=np.float64)
    wins = windows(n, W, stride)
    acc = None; wsum = np.zeros(n, dtype=np.float64)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        groups = defaultdict(list)
        for s, e in chunk:
            groups[e - s].append((s, e))
        for L, ws in groups.items():
            x = torch.tensor([doc.ids[s:e] for s, e in ws], dtype=torch.long, device=device)
            with amp():
                logits = net(x)
            probs = torch.softmax(logits.float(), -1).cpu().numpy()
            if acc is None:
                acc = np.zeros((n, probs.shape[-1]), dtype=np.float64)
            pos = np.arange(L)
            wt = 1.0 - np.abs((pos + 0.5) / L - 0.5) * 1.8          # 1.0 in the centre, 0.1 at the edges
            for (s, e), p in zip(ws, probs):
                acc[s:e] += p * wt[:, None]; wsum[s:e] += wt
    return np.log(acc / wsum[:, None] + 1e-9)


def decode_doc(doc, logp, I2L, ok, start_ok, use_viterbi=True):
    if logp.shape[0] == 0:
        return set()
    tags = viterbi(logp, ok, start_ok) if use_viterbi else logp.argmax(-1)
    out = set()
    for a, b, l in PT.bio_to_spans(list(tags), doc.spans, I2L):
        s, e = PT.norm_to_orig_span(doc.omap, a, b, doc.text)
        while s < e and doc.text[s].isspace(): s += 1
        while e > s and doc.text[e - 1].isspace(): e -= 1
        if e > s:
            out.add((s, e, l))
    return out


def propagate(text, spans, labels_on, min_len=4):
    """Document-level label propagation: a string predicted as label X somewhere in the document is also tagged X
    at its other exact, whole-word occurrences (if they do not overlap an existing prediction)."""
    if not labels_on:
        return set(spans)
    out = set(spans)
    taken = sorted((s, e) for s, e, _ in spans)
    vote = defaultdict(Counter)
    for s, e, l in spans:
        if e - s >= min_len:
            vote[text[s:e]][l] += 1
    occupied = np.zeros(len(text) + 1, dtype=bool)
    for s, e in taken:
        occupied[s:e] = True
    for surf, c in vote.items():
        l = c.most_common(1)[0][0]
        if l not in labels_on:
            continue
        for m in re.finditer(re.escape(surf), text):
            s, e = m.start(), m.end()
            if (s > 0 and text[s - 1].isalnum()) or (e < len(text) and text[e].isalnum()):
                continue
            if occupied[s:e].any():
                continue
            out.add((s, e, l)); occupied[s:e] = True
    return out


# ================================================================================================================
def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run_name", required=True)
    p.add_argument("--base_run", required=True, help="Phase-1 run, e.g. dapt_llama / explore_gpt2")
    p.add_argument("--base_file", default="best_model.pt")
    p.add_argument("--method", default="full", choices=["full", "head", "lora", "lorma", "bitfit"])
    p.add_argument("--kd_teacher", default="", help="Phase-2 run name of the teacher (enables distillation)")
    p.add_argument("--kd_alpha", type=float, default=0.5, help="weight of the gold-label loss (1-alpha: teacher)")
    p.add_argument("--kd_T", type=float, default=2.0)
    p.add_argument("--bidir", type=int, default=1, help="1 = bidirectional attention in decoder backbones")
    p.add_argument("--head", default="linear", choices=["linear", "bilstm"])
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--alpha", type=float, default=32)
    p.add_argument("--targets", default="q,k,v,o")
    p.add_argument("--win", type=int, default=512, help="training window (tokens)")
    p.add_argument("--infer_win", type=int, default=1024)
    p.add_argument("--infer_stride", type=int, default=512)
    p.add_argument("--epochs", type=int, default=20)
    p.add_argument("--bs", type=int, default=16)
    p.add_argument("--lr", type=float, default=5e-5, help="backbone LR (full / bitfit)")
    p.add_argument("--peft_lr", type=float, default=1e-3, help="adapter LR (lora / lorma)")
    p.add_argument("--head_lr", type=float, default=1e-3)
    p.add_argument("--wd", type=float, default=0.01)
    p.add_argument("--warmup", type=float, default=0.06)
    p.add_argument("--class_weight", default="sqrt", choices=["none", "sqrt"])
    p.add_argument("--patience", type=int, default=6)
    p.add_argument("--seed", type=int, default=13)
    p.add_argument("--raw_dir", required=True)
    p.add_argument("--split", required=True)
    p.add_argument("--tok", required=True)
    p.add_argument("--ckpt_dir", required=True, help="where Phase-1 checkpoints are (runs/<run>/<file>)")
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--max_steps", type=int, default=-1, help="debug")
    p.add_argument("--train_all", type=int, default=0,
                   help="1 = train on train+val documents for exactly --epochs epochs (no early stopping; final model)")
    p.add_argument("--prop_labels", default="", help="with --train_all: labels to propagate (chosen earlier on val)")
    p.add_argument("--save_logps", type=int, default=1, help="save token log-probabilities of val/test (for ensembling)")
    return p.parse_args(argv)


def hf_up(repo, local, remote):
    if not repo:
        return
    from huggingface_hub import HfApi
    for a in range(3):
        try:
            HfApi(token=os.environ.get("HF_TOKEN")).upload_file(path_or_fileobj=local, path_in_repo=remote,
                                                                 repo_id=repo, repo_type="model")
            return
        except Exception as e:
            print("[hf] upload failed:", e, flush=True); time.sleep(15 * (a + 1))


def get_ckpt(args, run, fname):
    local = os.path.join(args.ckpt_dir, "runs", run, fname)
    if not os.path.exists(local) and args.hf_repo:
        from huggingface_hub import hf_hub_download
        hf_hub_download(args.hf_repo, f"runs/{run}/{fname}", repo_type="model", local_dir=args.ckpt_dir,
                        token=os.environ.get("HF_TOKEN"))
    return local


def build_net(lm_cfg, n_tags, a):
    lm = LegalLM(ModelConfig(**lm_cfg))
    return NERNet(lm, n_tags, bidir=bool(a["bidir"]), head=a["head"])


def setup_method(net, a):
    for p in net.parameters():
        p.requires_grad = True
    m = a["method"]
    if m in ("head", "lora", "lorma", "bitfit"):
        for p in net.lm.parameters():
            p.requires_grad = False
    if m in ("lora", "lorma"):
        root = net.lm.enc_blocks if net.arch == "encdec" else net.lm.blocks
        n = add_adapters(root, m, set(a["targets"].split(",")), a["rank"], a["alpha"], 0.05)
        print(f"{m}: wrapped {n} linear layers", flush=True)
    if m == "bitfit":
        for name, p in net.lm.named_parameters():
            if name.endswith("bias") or ".n1." in name or ".n2." in name or "norm" in name:
                p.requires_grad = True


def main(argv=None):
    args = parse_args(argv)
    torch.manual_seed(args.seed); random.seed(args.seed); np.random.seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cuda = dev.type == "cuda"
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if cuda else (lambda: torch.autocast("cpu", enabled=False))
    run_dir = os.path.join(args.out_dir, args.run_name); os.makedirs(run_dir, exist_ok=True)
    logf = open(os.path.join(run_dir, "log.txt"), "a")

    def log(*m):
        s = f"[{time.strftime('%H:%M:%S')}][{args.run_name}] " + " ".join(str(x) for x in m)
        print(s, flush=True); logf.write(s + "\n"); logf.flush()

    # ---------------- data ----------------
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tok)
    LABELS = [x.strip() for x in open(os.path.join(args.raw_dir, "labels.txt"), encoding="utf-8") if x.strip()]
    L2I = {l: i for i, l in enumerate(LABELS)}; I2L = {i: l for l, i in L2I.items()}
    n_tags = 2 * len(LABELS) + 1
    texts, gold = {}, {}
    for f in ["fold1", "fold2"]:
        texts.update(PT.load_texts(os.path.join(args.raw_dir, f"{f}-text.json")))
        gold.update(PT.load_gold(os.path.join(args.raw_dir, f"{f}-gold.json")))
    test_texts = PT.load_texts(os.path.join(args.raw_dir, "fold3-text.json"))
    split = json.load(open(args.split))
    t0 = time.time()
    train_docs = [Doc(d, texts[d], tok, gold[d], L2I) for d in split["train"]]
    val_docs = [Doc(d, texts[d], tok, gold[d], L2I) for d in split["val"]]
    if args.train_all:
        train_docs = train_docs + val_docs                # final model: every labelled document
    test_docs = [Doc(d, test_texts[d], tok) for d in sorted(test_texts)]
    for name_, ds in [("train", train_docs), ("val", val_docs), ("test", test_docs)]:
        empty = [d.id for d in ds if len(d.ids) < 2]
        if empty:
            log(f"note: {len(empty)} {name_} document(s) with <2 tokens (empty text) -> predicted as no entities: {empty[:5]}")
    log(f"docs train {len(train_docs)} val {len(val_docs)} test {len(test_docs)} | train tokens "
        f"{sum(len(d.ids) for d in train_docs):,} | prep {time.time() - t0:.0f}s")

    # ---------------- model ----------------
    ck = torch.load(get_ckpt(args, args.base_run, args.base_file), map_location="cpu", weights_only=False)
    lm_cfg = ck["config"]
    a = vars(args)
    net = build_net(lm_cfg, n_tags, a)
    missing, unexpected = net.lm.load_state_dict(ck["model"], strict=False)
    bad = [k for k in missing if not k.startswith("blocks.")] if net.arch == "encdec" else list(missing)
    assert not bad, f"missing backbone weights: {bad[:5]}"
    del ck
    setup_method(net, a)
    if args.infer_win > lm_cfg["max_seq_len"] and lm_cfg["arch"] == "gpt2":
        args.infer_win = lm_cfg["max_seq_len"]
    net.to(dev)
    total, trainable = count_params(net)
    log(f"backbone {args.base_run} ({lm_cfg['arch']}) | method {args.method}{' + KD from ' + args.kd_teacher if args.kd_teacher else ''}"
        f" | params total {total / 1e6:.2f}M trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")

    teacher = None
    if args.kd_teacher:
        tpath = os.path.join(args.out_dir, args.kd_teacher, "best.pt")
        if not os.path.exists(tpath) and args.hf_repo:
            from huggingface_hub import hf_hub_download
            hf_hub_download(args.hf_repo, f"phase2/runs/{args.kd_teacher}/best.pt", repo_type="model",
                            local_dir=os.path.join(args.out_dir, "_teacher"), token=os.environ.get("HF_TOKEN"))
            tpath = os.path.join(args.out_dir, "_teacher", "phase2", "runs", args.kd_teacher, "best.pt")
        tck = torch.load(tpath, map_location="cpu", weights_only=False)
        teacher = build_net(tck["lm_config"], n_tags, tck["args"])
        setup_method(teacher, tck["args"])
        teacher.load_state_dict(tck["state_dict"])
        teacher.to(dev).eval()
        for p in teacher.parameters():
            p.requires_grad = False
        tt, _ = count_params(teacher)
        log(f"teacher {args.kd_teacher}: {tt / 1e6:.2f}M params (student {total / 1e6:.2f}M = {100 * total / tt:.0f}% of teacher)")
        del tck

    # ---------------- loss weights ----------------
    cnt = Counter(t for d in train_docs for t in d.tags)
    if args.class_weight == "sqrt":
        w = torch.tensor([math.sqrt(cnt[0] / max(1, cnt[t])) for t in range(n_tags)], dtype=torch.float)
        w = w.clamp(1.0, 10.0); w[0] = 1.0
    else:
        w = torch.ones(n_tags)
    w = w.to(dev)

    # ---------------- optimiser ----------------
    head_params = [p for n_, p in net.named_parameters() if p.requires_grad and not n_.startswith("lm.")]
    body_params = [p for n_, p in net.named_parameters() if p.requires_grad and n_.startswith("lm.")]
    body_lr = args.peft_lr if args.method in ("lora", "lorma") else args.lr
    groups = [{"params": head_params, "lr": args.head_lr}]
    if body_params:
        groups.append({"params": body_params, "lr": body_lr})
    opt = torch.optim.AdamW(groups, weight_decay=args.wd)
    base_lrs = [g["lr"] for g in groups]
    rng = random.Random(args.seed)
    steps_per_epoch = len(train_batches(train_docs, args.win, args.bs, random.Random(0)))
    total_steps = steps_per_epoch * args.epochs
    warm = max(1, int(args.warmup * total_steps))
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    ok, start_ok = allowed_transitions(len(LABELS))
    log(f"steps/epoch {steps_per_epoch} | total {total_steps} | warmup {warm} | lr body {body_lr} head {args.head_lr}")

    def evaluate(docs):
        net.eval()
        logps = {d.id: predict_logprobs(net, d, args.infer_win, args.infer_stride, dev, amp) for d in docs}
        net.train()
        return logps

    def decode_all(docs, logps, use_viterbi=True):
        return {d.id: decode_doc(d, logps[d.id], I2L, ok, start_ok, use_viterbi) for d in docs}

    def score(docs, logps, use_viterbi=True, prop_labels=(), decoded=None):
        dec = decoded if decoded is not None else decode_all(docs, logps, use_viterbi)
        pred = {d.id: propagate(d.text, dec[d.id], prop_labels) for d in docs}
        return PT.strict_macro({d.id: d.gold for d in docs}, pred, LABELS), pred

    # ---------------- training ----------------
    best, best_ep, bad_eps, step, hist = -1.0, -1, 0, 0, []
    t_train = time.time()
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    net.train()
    for ep in range(1, args.epochs + 1):
        tl, tn = 0.0, 0
        for batch in train_batches(train_docs, args.win, args.bs, rng):
            lr_mult = min(1.0, (step + 1) / warm) * (0.5 * (1 + math.cos(math.pi * min(1.0, step / total_steps))))
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * max(lr_mult, 0.02)
            x = torch.tensor([train_docs[di].ids[s:e] for di, s, e in batch], dtype=torch.long, device=dev)
            y = torch.tensor([train_docs[di].tags[s:e] for di, s, e in batch], dtype=torch.long, device=dev)
            with amp():
                logits = net(x)
            loss = F.cross_entropy(logits.view(-1, n_tags), y.view(-1), weight=w)
            if teacher is not None:
                with torch.no_grad(), amp():
                    tlog = teacher(x).float()
                T = args.kd_T
                kd = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(tlog / T, -1), reduction="batchmean") * T * T
                kd = kd / logits.shape[1]                     # batchmean over (B) -> per token
                loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update()
            tl += loss.item(); tn += 1; step += 1
            if args.max_steps > 0 and step >= args.max_steps:
                break
        if args.train_all:                                 # val is part of training data: no selection, keep last epoch
            hist.append({"epoch": ep, "loss": tl / max(1, tn)})
            log(f"epoch {ep:2d} loss {tl / max(1, tn):.4f} (train_all: no validation) | {(time.time() - t_train) / 60:.1f} min")
            best_ep = ep
            torch.save({"state_dict": net.state_dict(), "lm_config": lm_cfg, "args": a, "labels": LABELS,
                        "epoch": ep, "val_f1": None}, os.path.join(run_dir, "best.pt"))
            if args.max_steps > 0 and step >= args.max_steps:
                break
            continue
        logps = evaluate(val_docs)
        res, _ = score(val_docs, logps)
        hist.append({"epoch": ep, "loss": tl / max(1, tn), "val_f1": res["f1"], "val_p": res["p"], "val_r": res["r"]})
        log(f"epoch {ep:2d} loss {tl / max(1, tn):.4f} | val strict macro-F1 {res['f1']:.4f} (P {res['p']:.4f} R {res['r']:.4f})"
            f" | {(time.time() - t_train) / 60:.1f} min")
        if res["f1"] > best:
            best, best_ep, bad_eps = res["f1"], ep, 0
            torch.save({"state_dict": net.state_dict(), "lm_config": lm_cfg, "args": a, "labels": LABELS,
                        "epoch": ep, "val_f1": best}, os.path.join(run_dir, "best.pt"))
        else:
            bad_eps += 1
            if bad_eps >= args.patience:
                log(f"early stop (no improvement for {args.patience} epochs)"); break
        if args.max_steps > 0 and step >= args.max_steps:
            break
    train_min = (time.time() - t_train) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if cuda else 0.0

    # ---------------- final: best checkpoint, choose decoding on val, predict test ----------------
    net.load_state_dict(torch.load(os.path.join(run_dir, "best.pt"), map_location=dev, weights_only=False)["state_dict"])
    t_inf = time.time()
    val_logps = evaluate(val_docs)
    infer_val_s = time.time() - t_inf
    res_greedy, _ = score(val_docs, val_logps, use_viterbi=False)
    vdec = decode_all(val_docs, val_logps, True)
    res_vit, _ = score(val_docs, val_logps, decoded=vdec)
    prop_on = []                                           # per-label choice of propagation, judged on val
    for l in ([] if args.train_all else LABELS):
        r_on, _ = score(val_docs, val_logps, prop_labels=tuple(prop_on + [l]), decoded=vdec)
        r_off, _ = score(val_docs, val_logps, prop_labels=tuple(prop_on), decoded=vdec)
        if r_on["per_label"][l][2] > r_off["per_label"][l][2] + 1e-9:
            prop_on.append(l)
    if args.train_all:
        prop_on = [l for l in args.prop_labels.split(",") if l in L2I]
    res_final, val_pred = score(val_docs, val_logps, prop_labels=tuple(prop_on), decoded=vdec)
    if args.train_all:
        log("NOTE: train_all model - the val numbers below are on TRAINING documents (not a fair score)")
    log(f"VAL best epoch {best_ep}: greedy {res_greedy['f1']:.4f} | viterbi {res_vit['f1']:.4f} | "
        f"viterbi+propagation{prop_on} {res_final['f1']:.4f}")
    log("VAL per label F1: " + " ".join(f"{l}={res_final['per_label'][l][2]:.3f}" for l in LABELS))

    t_inf = time.time()
    test_logps = evaluate(test_docs)
    infer_test_s = time.time() - t_inf
    test_pred = {d.id: sorted(propagate(d.text, decode_doc(d, test_logps[d.id], I2L, ok, start_ok, True), tuple(prop_on)))
                 for d in test_docs}
    sub = {d: [[s, e, l] for s, e, l in v] for d, v in test_pred.items()}
    # ---- validate exactly like evaluate_phase2.py before writing ----
    assert set(sub) == set(test_texts), "test ids mismatch"
    for d, v in sub.items():
        seen = set()
        for s, e, l in v:
            assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(test_texts[d]) and l in L2I
            assert (s, e, l) not in seen; seen.add((s, e, l))
    pj = os.path.join(run_dir, "prediction.json")
    json.dump(sub, open(pj, "w"))
    zp = os.path.join(run_dir, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    json.dump({d: [list(x) for x in sorted(v)] for d, v in val_pred.items()}, open(os.path.join(run_dir, "val_prediction.json"), "w"))
    if args.save_logps:                                    # token log-probabilities for ensembling (float16)
        np.savez_compressed(os.path.join(run_dir, "logps.npz"),
                            **{f"val__{k}": v.astype(np.float16) for k, v in val_logps.items()},
                            **{f"test__{k}": v.astype(np.float16) for k, v in test_logps.items()})
    n_test_ents = sum(len(v) for v in sub.values())
    summary = {"run": args.run_name, "base_run": args.base_run, "arch": lm_cfg["arch"], "method": args.method,
               "kd_teacher": args.kd_teacher or None, "bidir": args.bidir, "head": args.head,
               "params_total": total, "params_trainable": trainable, "best_epoch": best_ep,
               "val_f1": res_final["f1"], "val_p": res_final["p"], "val_r": res_final["r"],
               "val_f1_greedy": res_greedy["f1"], "val_f1_viterbi": res_vit["f1"], "propagation_labels": prop_on,
               "val_per_label": {l: res_final["per_label"][l] for l in LABELS},
               "train_minutes": round(train_min, 2), "peak_gpu_gb": round(peak_gb, 2),
               "infer_val_seconds": round(infer_val_s, 1), "infer_test_seconds": round(infer_test_s, 1),
               "test_entities": n_test_ents, "history": hist, "train_all": bool(args.train_all), "args": a}
    json.dump(summary, open(os.path.join(run_dir, "summary.json"), "w"), indent=1)
    log(f"TEST: {n_test_ents} entities in {len(sub)} docs -> {zp}")
    for fn in ["summary.json", "prediction.json", "submission.zip", "val_prediction.json", "log.txt", "logps.npz", "best.pt"]:
        if os.path.exists(os.path.join(run_dir, fn)):
            hf_up(args.hf_repo, os.path.join(run_dir, fn), f"phase2/runs/{args.run_name}/{fn}")
    log("DONE")


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/p2ner.py


In [27]:
%%writefile /kaggle/working/p2ens.py
"""CS-781 Phase 2: ensemble of our fine-tuned NER models (all built on our own Phase-1 models).

For every candidate run we need its token probabilities on the validation and test documents (logps.npz; computed
from the run's best.pt if missing). Ensemble = average of the models' probabilities, then the same constrained
Viterbi decoding + label propagation as single models. Members are chosen greedily on the validation set.

Outputs (in out_dir):
  ens_val/  : the chosen models exactly as validated                       -> submission.zip
  ens_all/  : same members, but each replaced by its full-data twin(s) (<run>_all*) when they exist -> submission.zip
"""
import argparse, json, os, sys, time, zipfile
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import p2ner as N
import p2text as PT


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--raw_dir", required=True)
    p.add_argument("--split", required=True)
    p.add_argument("--tok", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--min_val", type=float, default=0.30, help="only runs with val F1 >= this are candidates")
    p.add_argument("--max_members", type=int, default=6)
    p.add_argument("--min_gain", type=float, default=0.002)
    return p.parse_args(argv)


def hf_get(repo, remote, local_dir):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def main(argv=None):
    args = parse_args(argv)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if dev.type == "cuda" else (lambda: torch.autocast("cpu", enabled=False))
    t0 = time.time()

    def log(*m):
        print(f"[{time.strftime('%H:%M:%S')}][ensemble] " + " ".join(str(x) for x in m), flush=True)

    # ---------------- data ----------------
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tok)
    LABELS = [x.strip() for x in open(os.path.join(args.raw_dir, "labels.txt"), encoding="utf-8") if x.strip()]
    L2I = {l: i for i, l in enumerate(LABELS)}; I2L = {i: l for l, i in L2I.items()}
    n_tags = 2 * len(LABELS) + 1
    texts, gold = {}, {}
    for f in ["fold1", "fold2"]:
        texts.update(PT.load_texts(os.path.join(args.raw_dir, f"{f}-text.json")))
        gold.update(PT.load_gold(os.path.join(args.raw_dir, f"{f}-gold.json")))
    test_texts = PT.load_texts(os.path.join(args.raw_dir, "fold3-text.json"))
    split = json.load(open(args.split))
    val_docs = [N.Doc(d, texts[d], tok, gold[d], L2I) for d in split["val"]]
    test_docs = [N.Doc(d, test_texts[d], tok) for d in sorted(test_texts)]
    ok, start_ok = N.allowed_transitions(len(LABELS))
    truth = {d.id: d.gold for d in val_docs}

    # ---------------- which runs exist ----------------
    summaries = {}
    names = set(x for x in os.listdir(args.out_dir) if os.path.isdir(os.path.join(args.out_dir, x)))
    if args.hf_repo:
        from huggingface_hub import HfApi
        for f in HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(args.hf_repo):
            if f.startswith("phase2/runs/") and f.endswith("/summary.json"):
                names.add(f.split("/")[2])
    for n in sorted(names):
        if n.startswith("ens_") or n.startswith("_"):
            continue
        p = os.path.join(args.out_dir, n, "summary.json")
        if not os.path.exists(p):
            q = hf_get(args.hf_repo, f"phase2/runs/{n}/summary.json", os.path.join(args.out_dir, "_hub"))
            if q is None:
                continue
            os.makedirs(os.path.dirname(p), exist_ok=True)
            import shutil; shutil.copy(q, p)
        summaries[n] = json.load(open(p))
    cands = sorted([n for n, s in summaries.items() if not s.get("train_all") and s["val_f1"] >= args.min_val],
                   key=lambda n: -summaries[n]["val_f1"])
    twins = {n: sorted(t for t, s in summaries.items() if s.get("train_all") and t.startswith(n + "_all")) for n in cands}
    log(f"{len(cands)} candidate runs (val F1 >= {args.min_val}):", ", ".join(f"{n}={summaries[n]['val_f1']:.3f}" for n in cands))
    log("full-data twins:", {n: t for n, t in twins.items() if t} or "none")

    # ---------------- token probabilities for every needed run ----------------
    def get_probs(run):
        path = os.path.join(args.out_dir, run, "logps.npz")
        if not os.path.exists(path):
            q = hf_get(args.hf_repo, f"phase2/runs/{run}/logps.npz", os.path.join(args.out_dir, "_hub"))
            if q is not None:
                import shutil; shutil.copy(q, path)
        if not os.path.exists(path):                       # older run: compute from its best.pt
            bp = os.path.join(args.out_dir, run, "best.pt")
            if not os.path.exists(bp):
                q = hf_get(args.hf_repo, f"phase2/runs/{run}/best.pt", os.path.join(args.out_dir, "_hub"))
                bp = q
            ck = torch.load(bp, map_location="cpu", weights_only=False)
            a = ck["args"]
            net = N.build_net(ck["lm_config"], n_tags, a)
            N.setup_method(net, a)
            net.load_state_dict(ck["state_dict"])
            net.to(dev).eval()
            W, S = a["infer_win"], a["infer_stride"]
            if ck["lm_config"]["arch"] == "gpt2":                 # learned positions: never exceed the trained length
                W = min(W, ck["lm_config"]["max_seq_len"]); S = min(S, W)
            vl = {d.id: N.predict_logprobs(net, d, W, S, dev, amp) for d in val_docs}
            tl = {d.id: N.predict_logprobs(net, d, W, S, dev, amp) for d in test_docs}
            np.savez_compressed(path, **{f"val__{k}": v.astype(np.float16) for k, v in vl.items()},
                                **{f"test__{k}": v.astype(np.float16) for k, v in tl.items()})
            del net, ck
            if dev.type == "cuda":
                torch.cuda.empty_cache()
            log(f"computed probabilities for {run}")
        z = np.load(path)
        val = {d.id: np.exp(z[f"val__{d.id}"].astype(np.float32)) for d in val_docs}
        test = {d.id: np.exp(z[f"test__{d.id}"].astype(np.float32)) for d in test_docs}
        return val, test

    P = {}
    for n in cands + [t for n in cands for t in twins[n]]:
        P[n] = get_probs(n)

    def mean_probs(dicts):
        return {k: sum(d[k] for d in dicts) / len(dicts) for k in dicts[0]}

    def decode(docs, probs, prop=()):
        return {d.id: N.propagate(d.text, N.decode_doc(d, np.log(probs[d.id] + 1e-9), I2L, ok, start_ok, True), prop)
                for d in docs}

    def val_f1(members, prop=()):
        return PT.strict_macro(truth, decode(val_docs, mean_probs([P[m][0] for m in members]), prop), LABELS)

    # ---------------- greedy forward selection on validation ----------------
    single = {n: val_f1([n])["f1"] for n in cands}
    log("single-model val F1 (no propagation):", ", ".join(f"{n}={v:.4f}" for n, v in sorted(single.items(), key=lambda x: -x[1])))
    chosen = [max(single, key=single.get)]; cur = single[chosen[0]]
    steps = [(list(chosen), cur)]
    while len(chosen) < args.max_members:
        best_n, best_f = None, cur
        for n in cands:
            if n in chosen:
                continue
            f = val_f1(chosen + [n])["f1"]
            if f > best_f:
                best_n, best_f = n, f
        if best_n is None or best_f - cur < args.min_gain:
            break
        chosen.append(best_n); cur = best_f; steps.append((list(chosen), cur))
        log(f"  + {best_n:22s} -> val F1 {cur:.4f}")
    # propagation per label, judged on validation (same rule as single models)
    vprobs = mean_probs([P[m][0] for m in chosen])
    vdec = {d.id: N.decode_doc(d, np.log(vprobs[d.id] + 1e-9), I2L, ok, start_ok, True) for d in val_docs}

    def f_with(prop):
        pred = {d.id: N.propagate(d.text, vdec[d.id], tuple(prop)) for d in val_docs}
        return PT.strict_macro(truth, pred, LABELS)
    prop_on = []
    for l in LABELS:
        if f_with(prop_on + [l])["per_label"][l][2] > f_with(prop_on)["per_label"][l][2] + 1e-9:
            prop_on.append(l)
    final = f_with(prop_on)
    log(f"ENSEMBLE {chosen} -> val F1 {final['f1']:.4f} (P {final['p']:.4f} R {final['r']:.4f}) | propagation {prop_on}")
    log("per label: " + " ".join(f"{l}={final['per_label'][l][2]:.3f}" for l in LABELS))

    # ---------------- write the two submissions ----------------
    def write(name, members_test):
        tprobs = mean_probs(members_test)
        pred = decode(test_docs, tprobs, tuple(prop_on))
        sub = {d: [[s, e, l] for s, e, l in sorted(v)] for d, v in pred.items()}
        assert set(sub) == set(test_texts)
        for d, v in sub.items():
            seen = set()
            for s, e, l in v:
                assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(test_texts[d]) and l in L2I
                assert (s, e, l) not in seen; seen.add((s, e, l))
        rd = os.path.join(args.out_dir, name); os.makedirs(rd, exist_ok=True)
        json.dump(sub, open(os.path.join(rd, "prediction.json"), "w"))
        with zipfile.ZipFile(os.path.join(rd, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(os.path.join(rd, "prediction.json"), arcname="prediction.json")
        return rd, sum(len(v) for v in sub.values())

    info = {"members": chosen, "val_f1": final["f1"], "val_p": final["p"], "val_r": final["r"], "propagation_labels": prop_on,
            "val_per_label": final["per_label"], "selection_steps": steps, "single_val_f1": single}
    rd, ne = write("ens_val", [P[m][1] for m in chosen])
    json.dump({**info, "test_members": chosen}, open(os.path.join(rd, "summary.json"), "w"), indent=1)
    log(f"ens_val: {ne} test entities -> {rd}/submission.zip")
    outs = ["ens_val"]
    if any(twins[m] for m in chosen):
        per_member = []
        used = []
        for m in chosen:
            if twins[m]:
                per_member.append(mean_probs([P[t][1] for t in twins[m]])); used.append(twins[m])
            else:
                per_member.append(P[m][1]); used.append([m])
        rd, ne = write("ens_all", per_member)
        json.dump({**info, "test_members": used}, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        log(f"ens_all (full-data twins {used}): {ne} test entities -> {rd}/submission.zip")
        outs.append("ens_all")
    if args.hf_repo:
        for o in outs:
            for fn in ["summary.json", "prediction.json", "submission.zip"]:
                N.hf_up(args.hf_repo, os.path.join(args.out_dir, o, fn), f"phase2/runs/{o}/{fn}")
    log(f"DONE in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/p2ens.py


In [32]:
%%writefile /kaggle/working/p2ens.py
"""CS-781 Phase 2: ensemble of our fine-tuned NER models (all built on our own Phase-1 models).

For every candidate run we need its token probabilities on the validation and test documents (logps.npz; computed
from the run's best.pt if missing). Ensemble = average of the models' probabilities, then the same constrained
Viterbi decoding + label propagation as single models. Members are chosen greedily on the validation set.

Outputs (in out_dir):
  ens_val/  : the chosen models exactly as validated                       -> submission.zip
  ens_all/  : same members, but each replaced by its full-data twin(s) (<run>_all*) when they exist -> submission.zip
"""
import argparse, json, os, sys, time, zipfile
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import p2ner as N
import p2text as PT


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--raw_dir", required=True)
    p.add_argument("--split", required=True)
    p.add_argument("--tok", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--min_val", type=float, default=0.30, help="only runs with val F1 >= this are candidates")
    p.add_argument("--max_members", type=int, default=6)
    p.add_argument("--min_gain", type=float, default=0.002)
    p.add_argument("--max_cands", type=int, default=10, help="only the top-N runs by val F1 are tried")
    return p.parse_args(argv)


def hf_get(repo, remote, local_dir):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def main(argv=None):
    args = parse_args(argv)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if dev.type == "cuda" else (lambda: torch.autocast("cpu", enabled=False))
    t0 = time.time()

    def log(*m):
        print(f"[{time.strftime('%H:%M:%S')}][ensemble] " + " ".join(str(x) for x in m), flush=True)

    # ---------------- data ----------------
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tok)
    LABELS = [x.strip() for x in open(os.path.join(args.raw_dir, "labels.txt"), encoding="utf-8") if x.strip()]
    L2I = {l: i for i, l in enumerate(LABELS)}; I2L = {i: l for l, i in L2I.items()}
    n_tags = 2 * len(LABELS) + 1
    texts, gold = {}, {}
    for f in ["fold1", "fold2"]:
        texts.update(PT.load_texts(os.path.join(args.raw_dir, f"{f}-text.json")))
        gold.update(PT.load_gold(os.path.join(args.raw_dir, f"{f}-gold.json")))
    test_texts = PT.load_texts(os.path.join(args.raw_dir, "fold3-text.json"))
    split = json.load(open(args.split))
    val_docs = [N.Doc(d, texts[d], tok, gold[d], L2I) for d in split["val"]]
    test_docs = [N.Doc(d, test_texts[d], tok) for d in sorted(test_texts)]
    ok, start_ok = N.allowed_transitions(len(LABELS))
    truth = {d.id: d.gold for d in val_docs}

    # ---------------- which runs exist ----------------
    summaries = {}
    names = set(x for x in os.listdir(args.out_dir) if os.path.isdir(os.path.join(args.out_dir, x)))
    if args.hf_repo:
        from huggingface_hub import HfApi
        for f in HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(args.hf_repo):
            if f.startswith("phase2/runs/") and f.endswith("/summary.json"):
                names.add(f.split("/")[2])
    for n in sorted(names):
        if n.startswith("ens_") or n.startswith("_"):
            continue
        p = os.path.join(args.out_dir, n, "summary.json")
        if not os.path.exists(p):
            q = hf_get(args.hf_repo, f"phase2/runs/{n}/summary.json", os.path.join(args.out_dir, "_hub"))
            if q is None:
                continue
            os.makedirs(os.path.dirname(p), exist_ok=True)
            import shutil; shutil.copy(q, p)
        summaries[n] = json.load(open(p))
    cands = sorted([n for n, s in summaries.items() if not s.get("train_all") and s["val_f1"] >= args.min_val],
                   key=lambda n: -summaries[n]["val_f1"])
    cands = cands[:args.max_cands]
    twins = {n: sorted(t for t, s in summaries.items() if s.get("train_all") and t.startswith(n + "_all")) for n in cands}
    log(f"{len(cands)} candidate runs (val F1 >= {args.min_val}):", ", ".join(f"{n}={summaries[n]['val_f1']:.3f}" for n in cands))
    log("full-data twins:", {n: t for n, t in twins.items() if t} or "none")

    # ---------------- token probabilities for every needed run ----------------
    def get_probs(run):
        path = os.path.join(args.out_dir, run, "logps.npz")
        if not os.path.exists(path):
            q = hf_get(args.hf_repo, f"phase2/runs/{run}/logps.npz", os.path.join(args.out_dir, "_hub"))
            if q is not None:
                import shutil; shutil.copy(q, path)
        if not os.path.exists(path):                       # older run: compute from its best.pt
            bp = os.path.join(args.out_dir, run, "best.pt")
            if not os.path.exists(bp):
                q = hf_get(args.hf_repo, f"phase2/runs/{run}/best.pt", os.path.join(args.out_dir, "_hub"))
                bp = q
            ck = torch.load(bp, map_location="cpu", weights_only=False)
            a = ck["args"]
            net = N.build_net(ck["lm_config"], n_tags, a)
            N.setup_method(net, a)
            net.load_state_dict(ck["state_dict"])
            net.to(dev).eval()
            W, S = a["infer_win"], a["infer_stride"]
            if ck["lm_config"]["arch"] == "gpt2":                 # learned positions: never exceed the trained length
                W = min(W, ck["lm_config"]["max_seq_len"]); S = min(S, W)
            vl = {d.id: N.predict_logprobs(net, d, W, S, dev, amp) for d in val_docs}
            tl = {d.id: N.predict_logprobs(net, d, W, S, dev, amp) for d in test_docs}
            np.savez_compressed(path, **{f"val__{k}": v.astype(np.float16) for k, v in vl.items()},
                                **{f"test__{k}": v.astype(np.float16) for k, v in tl.items()})
            del net, ck
            if dev.type == "cuda":
                torch.cuda.empty_cache()
            log(f"computed probabilities for {run}")
        z = np.load(path)
        val = {d.id: np.exp(z[f"val__{d.id}"].astype(np.float32)) for d in val_docs}
        test = {d.id: np.exp(z[f"test__{d.id}"].astype(np.float32)) for d in test_docs}
        return val, test

    P = {}
    for n in cands + [t for n in cands for t in twins[n]]:
        P[n] = get_probs(n)

    def mean_probs(dicts):
        return {k: sum(d[k] for d in dicts) / len(dicts) for k in dicts[0]}

    def decode(docs, probs, prop=()):
        return {d.id: N.propagate(d.text, N.decode_doc(d, np.log(probs[d.id] + 1e-9), I2L, ok, start_ok, True), prop)
                for d in docs}

    def full_score(members):
        """Score a member set EXACTLY like the final submission: Viterbi decode + per-label propagation chosen on val."""
        vprobs = mean_probs([P[m][0] for m in members])
        vdec = {d.id: N.decode_doc(d, np.log(vprobs[d.id] + 1e-9), I2L, ok, start_ok, True) for d in val_docs}

        def f_with(prop):
            pred = {d.id: N.propagate(d.text, vdec[d.id], tuple(prop)) for d in val_docs}
            return PT.strict_macro(truth, pred, LABELS)
        prop_on = []
        for l in LABELS:
            if f_with(prop_on + [l])["per_label"][l][2] > f_with(prop_on)["per_label"][l][2] + 1e-9:
                prop_on.append(l)
        return f_with(prop_on), prop_on

    # ---------------- greedy forward selection on validation (same scoring as the final result) ----------------
    single_full = {n: full_score([n]) for n in cands}
    single = {n: r[0]["f1"] for n, r in single_full.items()}
    log("single-model val F1 (with propagation):", ", ".join(f"{n}={v:.4f}" for n, v in sorted(single.items(), key=lambda x: -x[1])))
    chosen = [max(single, key=single.get)]; cur = single[chosen[0]]
    final, prop_on = single_full[chosen[0]]
    steps = [(list(chosen), cur)]
    while len(chosen) < args.max_members:
        best_n, best_f, best_r = None, cur, None
        for n in cands:
            if n in chosen:
                continue
            r = full_score(chosen + [n])
            if r[0]["f1"] > best_f:
                best_n, best_f, best_r = n, r[0]["f1"], r
        if best_n is None or best_f - cur < args.min_gain:
            break
        chosen.append(best_n); cur = best_f; final, prop_on = best_r; steps.append((list(chosen), cur))
        log(f"  + {best_n:22s} -> val F1 {cur:.4f}")
    assert final["f1"] >= max(single.values()) - 1e-12      # never worse than the best single model on val
    log(f"ENSEMBLE {chosen} -> val F1 {final['f1']:.4f} (P {final['p']:.4f} R {final['r']:.4f}) | propagation {prop_on}")
    log("per label: " + " ".join(f"{l}={final['per_label'][l][2]:.3f}" for l in LABELS))

    # ---------------- write the two submissions ----------------
    def write(name, members_test, prop=None):
        tprobs = mean_probs(members_test)
        pred = decode(test_docs, tprobs, tuple(prop_on if prop is None else prop))
        sub = {d: [[s, e, l] for s, e, l in sorted(v)] for d, v in pred.items()}
        assert set(sub) == set(test_texts)
        for d, v in sub.items():
            seen = set()
            for s, e, l in v:
                assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(test_texts[d]) and l in L2I
                assert (s, e, l) not in seen; seen.add((s, e, l))
        rd = os.path.join(args.out_dir, name); os.makedirs(rd, exist_ok=True)
        json.dump(sub, open(os.path.join(rd, "prediction.json"), "w"))
        with zipfile.ZipFile(os.path.join(rd, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(os.path.join(rd, "prediction.json"), arcname="prediction.json")
        return rd, sum(len(v) for v in sub.values())

    info = {"members": chosen, "val_f1": final["f1"], "val_p": final["p"], "val_r": final["r"], "propagation_labels": prop_on,
            "val_per_label": final["per_label"], "selection_steps": steps, "single_val_f1": single}
    rd, ne = write("ens_val", [P[m][1] for m in chosen])
    json.dump({**info, "test_members": chosen}, open(os.path.join(rd, "summary.json"), "w"), indent=1)
    log(f"ens_val: {ne} test entities -> {rd}/submission.zip")
    outs = ["ens_val"]
    if any(twins[m] for m in chosen):
        per_member = []
        used = []
        for m in chosen:
            if twins[m]:
                per_member.append(mean_probs([P[t][1] for t in twins[m]])); used.append(twins[m])
            else:
                per_member.append(P[m][1]); used.append([m])
        rd, ne = write("ens_all", per_member)
        json.dump({**info, "test_members": used}, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        log(f"ens_all (full-data twins {used}): {ne} test entities -> {rd}/submission.zip")
        outs.append("ens_all")
    # robust alternative: PLAIN average of the top-5 runs by their own val F1 (no greedy search, so it is less
    # fitted to our small 10-document validation set)
    top = cands[:5]
    if len(top) >= 2:
        r_top, prop_top = full_score(top)
        rd, ne = write("ens_top5", [P[m][1] for m in top], prop_top)
        json.dump({"members": top, "val_f1": r_top["f1"], "val_p": r_top["p"], "val_r": r_top["r"],
                   "propagation_labels": prop_top, "val_per_label": r_top["per_label"]},
                  open(os.path.join(rd, "summary.json"), "w"), indent=1)
        log(f"ens_top5 {top} -> val F1 {r_top['f1']:.4f} | propagation {prop_top} | {ne} test entities -> {rd}/submission.zip")
        outs.append("ens_top5")
    # bigger plain averages (test showed: plain averaging works, and full-data twins help inside an ensemble).
    # Propagation labels are always chosen on val with the ORIGINAL runs only (twins have seen the val documents).
    for K, with_twins in [(8, False), (5, True), (8, True)]:
        top = cands[:K]
        if len(top) < 2:
            continue
        name = f"ens_top{len(top)}" + ("_twins" if with_twins else "")
        if name in outs:
            continue
        r_k, prop_k = full_score(top)
        members_test = list(top) + ([t for m in top for t in twins[m]] if with_twins else [])
        rd, ne = write(name, [P[m][1] for m in members_test], prop_k)
        json.dump({"members_val": top, "test_members": members_test, "val_f1_of_originals": r_k["f1"],
                   "propagation_labels": prop_k}, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        log(f"{name}: {len(members_test)} models {members_test} | val F1 (originals) {r_k['f1']:.4f} | "
            f"propagation {prop_k} | {ne} test entities -> {rd}/submission.zip")
        outs.append(name)
    if args.hf_repo:
        for o in outs:
            for fn in ["summary.json", "prediction.json", "submission.zip"]:
                N.hf_up(args.hf_repo, os.path.join(args.out_dir, o, fn), f"phase2/runs/{o}/{fn}")
    log(f"DONE in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/p2ens.py


In [28]:
# ===================== P2 CELL 7 - Run the experiments on both GPUs (queue with dependencies, resumable) =====================
import threading, subprocess
from huggingface_hub import hf_hub_download

# one flat folder with the official files (p2ner.py expects fold1/fold2/fold3 + labels.txt side by side)
RAW = f"{P2}/flat"; os.makedirs(RAW, exist_ok=True)
for n in need:
    shutil.copy(LOCAL[n], f"{RAW}/{n}")
CK = "/tmp/cs781/ckpt"                    # Phase-1 checkpoints (downloaded once, before the workers start)
OUT = f"{WORK}/p2runs"; os.makedirs(OUT, exist_ok=True)
for run, fn in [("dapt_llama", "best_model.pt"), ("explore_gpt2", "final_model.pt"),
                ("explore_encdec", "final_model.pt"), ("explore_shared", "final_model.pt")]:
    hf_hub_download(CKPT_REPO, f"runs/{run}/{fn}", repo_type="model", local_dir=CK, token=HF_TOKEN)
print("Phase-1 checkpoints ready")

COMMON = dict(epochs=20, win=512, infer_win=1024, infer_stride=512, bs=16, lr=1e-4, head_lr=1e-3, peft_lr=1e-3,
              rank=16, alpha=32, patience=6)
LLAMA  = dict(base_run="dapt_llama", base_file="best_model.pt")
GPT2   = dict(base_run="explore_gpt2", base_file="final_model.pt")
ENCDEC = dict(base_run="explore_encdec", base_file="final_model.pt")
SHARED = dict(base_run="explore_shared", base_file="final_model.pt")
RUNS = [   # ---- the 4 required experiments (4 different Phase-1 models, 4 different methods) ----
    ("p2_llama_full",   {**LLAMA,  "method": "full"}),                                   # full fine-tuning (teacher)
    ("p2_gpt2_lorma",   {**GPT2,   "method": "lorma"}),                                  # LoRMA (compulsory)
    ("p2_encdec_lora",  {**ENCDEC, "method": "lora"}),                                   # LoRA on the encoder
    ("p2_shared_kd",    {**SHARED, "method": "full", "kd_teacher": "p2_llama_full"}),    # knowledge distillation
           # ---- extra experiments for the analysis ----
    ("p2_shared_full",  {**SHARED, "method": "full"}),                                   # same student WITHOUT KD
    ("p2_llama_lora",   {**LLAMA,  "method": "lora"}),                                   # PEFT vs full, same model
    ("p2_llama_lorma",  {**LLAMA,  "method": "lorma"}),
    ("p2_gpt2_full",    {**GPT2,   "method": "full"}),
    ("p2_llama_causal", {**LLAMA,  "method": "full", "bidir": 0}),                       # context: left-only attention
    ("p2_llama_head",   {**LLAMA,  "method": "head"}),                                   # frozen backbone, linear probe
]
RUNS += [   # ---- round 2 ----
    ("p2_gpt2_full_causal",   {**GPT2,   "method": "full", "bidir": 0}),             # GPT-2 fix: keep left-only attention
    ("p2_gpt2_full_lr3e5",    {**GPT2,   "method": "full", "lr": 3e-5}),             # GPT-2 fix: gentler learning rate
    ("p2_gpt2_lorma_causal",  {**GPT2,   "method": "lorma", "bidir": 0}),
    ("p2_shared_kd_lorma",    {**SHARED, "method": "full", "kd_teacher": "p2_llama_lorma"}),  # KD from the BEST teacher
    ("p2_llama_lorma_bilstm", {**LLAMA,  "method": "lorma", "head": "bilstm"}),      # different prediction head
    ("p2_llama_lorma_w1024",  {**LLAMA,  "method": "lorma", "win": 1024, "bs": 8}),  # longer training context
    ("p2_encdec_full",        {**ENCDEC, "method": "full"}),                         # EncDec: full vs LoRA
]
TWIN_TOPK = 4        # round 3: re-train the TOP-4 validated configurations on ALL 70 labelled documents
ONLY = None          # e.g. ["p2_llama_full", "p2_gpt2_lorma"] to run a subset

# ---- 2-minute GPU safety test: a few training steps + full val/test prediction + zip, nothing uploaded ----
smoke = [sys.executable, f"{WORK}/p2ner.py", "--run_name", "_smoke", "--raw_dir", RAW, "--split", f"{P2}/data/split.json",
         "--tok", TOK_PATH, "--ckpt_dir", CK, "--out_dir", f"{P2}/smoke", "--hf_repo", "",
         "--base_run", "explore_shared", "--base_file", "final_model.pt", "--method", "lora", "--epochs", "1", "--max_steps", "3"]
t_s = time.time()
r = subprocess.run(smoke, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "][_smoke]" in l][-6:]))
if r.returncode != 0 or not os.path.exists(f"{P2}/smoke/_smoke/submission.zip"):
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    raise RuntimeError("SAFETY TEST FAILED - nothing was trained. Send this output to Claude.")
print(f"safety test passed in {time.time() - t_s:.0f}s")
if torch.cuda.device_count() == 0:
    print("\nCPU DRY RUN PASSED on the real data. Now switch Accelerator to GPU T4 x2 and use Save Version -> Save & Run All.")
    RUNS = []                                                  # no training on CPU
else:
    print("starting the experiments\n")

hub_files = set(api.list_repo_files(CKPT_REPO))

# ---- round 3 (automatic): full-data "twins" of the best validated runs finished in EARLIER sessions ----
_summ = {}
for f in hub_files:
    if f.startswith("phase2/runs/") and f.endswith("/summary.json"):
        n = f.split("/")[2]
        if n.startswith("ens_") or n.startswith("_"): continue
        s_ = json.load(open(hf_hub_download(CKPT_REPO, f, repo_type="model", local_dir=f"{P2}/hub", token=HF_TOKEN)))
        if not s_.get("train_all"): _summ[n] = s_
KEEP = ["base_run", "base_file", "method", "kd_teacher", "bidir", "head", "rank", "alpha", "targets", "win", "infer_win",
        "infer_stride", "bs", "lr", "peft_lr", "head_lr", "class_weight"]
for rank_i, n in enumerate(sorted(_summ, key=lambda k: -_summ[k]["val_f1"])[:TWIN_TOPK if torch.cuda.device_count() else 0]):
    a_ = _summ[n]["args"]
    cfg_ = {k: a_[k] for k in KEEP if k in a_ and a_[k] not in (None, "")}
    cfg_.update(train_all=1, epochs=max(3, _summ[n]["best_epoch"]), patience=999,
                prop_labels=",".join(_summ[n]["propagation_labels"]))
    for sd in ([13, 14] if rank_i == 0 else [13]):           # the best configuration gets a 2nd seed
        RUNS.append((f"{n}_all" + ("" if sd == 13 else f"_s{sd}"), {**cfg_, "seed": sd}))
        print(f"round 3 twin: {RUNS[-1][0]:28s} (val F1 of {n}: {_summ[n]['val_f1']:.4f}, epochs {cfg_['epochs']})")
done, failed, lock = set(), set(), threading.Lock()
pending = [(n, c) for n, c in RUNS if ONLY is None or n in ONLY]
pending.sort(key=lambda x: 0 if x[1].get("train_all") else 1)   # full-data twins first (most useful for the test score)
for n, c in list(pending):
    if f"phase2/runs/{n}/summary.json" in hub_files:          # finished in an earlier session -> skip
        os.makedirs(f"{OUT}/{n}", exist_ok=True)
        for fn in ["summary.json", "submission.zip", "prediction.json"]:
            if f"phase2/runs/{n}/{fn}" in hub_files:
                p = hf_hub_download(CKPT_REPO, f"phase2/runs/{n}/{fn}", repo_type="model", local_dir=f"{P2}/hub", token=HF_TOKEN)
                shutil.copy(p, f"{OUT}/{n}/{fn}")
        done.add(n); pending.remove((n, c)); print("already done:", n)

running = set()

def next_job():
    with lock:
        for i, (n, c) in enumerate(pending):
            t = c.get("kd_teacher")
            if t and t not in done and f"phase2/runs/{t}/best.pt" in hub_files:
                done.add(t)                                   # teacher finished in an earlier session (on HF)
            if t and t not in done and t not in running and all(t != m for m, _ in pending):
                failed.add(t)                                 # teacher is not available and not scheduled
            if t and t in failed:
                pending.pop(i); failed.add(n); print(f"SKIP {n}: teacher {t} failed / not available", flush=True); return "AGAIN"
            if not t or t in done:
                running.add(n); return pending.pop(i)
        return "WAIT" if pending else None

def worker(gpu):
    while True:
        job = next_job()
        if job is None: return
        if job in ("WAIT", "AGAIN"):
            time.sleep(30 if job == "WAIT" else 0); continue
        name, cfg = job
        cmd = [sys.executable, f"{WORK}/p2ner.py", "--run_name", name, "--raw_dir", RAW, "--split", f"{P2}/data/split.json",
               "--tok", TOK_PATH, "--ckpt_dir", CK, "--out_dir", OUT, "--hf_repo", CKPT_REPO]
        for k, v in {**COMMON, **cfg}.items():
            cmd += [f"--{k}", str(v)]
        with open(f"{OUT}/{name}.out", "w") as f:
            r = subprocess.run(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=f, stderr=subprocess.STDOUT)
        ok_ = r.returncode == 0 and os.path.exists(f"{OUT}/{name}/summary.json")
        with lock:
            (done if ok_ else failed).add(name); running.discard(name)
            print(f"{'FINISHED' if ok_ else 'FAILED  '} {name} on GPU {gpu}", flush=True)
            if not ok_:                                   # show the reason right here, in the notebook log
                print(f"===== last lines of {name}.out =====\n" + "\n".join(open(f"{OUT}/{name}.out").read().splitlines()[-30:]), flush=True)

def status_line(name):
    p = f"{OUT}/{name}.out"
    if not os.path.exists(p): return "(waiting)"
    keep = [l for l in open(p).read().splitlines() if "epoch" in l or "VAL" in l or "Error" in l or "DONE" in l]
    return keep[-1][:170] if keep else "(starting)"

threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads: t.start()
t0 = time.time()
while any(t.is_alive() for t in threads):
    for t in threads: t.join(timeout=300 / len(threads))
    print(f"--- {(time.time() - t0) / 60:.0f} min | done {sorted(done)} | failed {sorted(failed)}", flush=True)
    for n, _ in RUNS:
        if n not in done and (ONLY is None or n in ONLY): print(f"   {n:16s} {status_line(n)}", flush=True)
print("ALL FINISHED. done:", sorted(done), "| failed:", sorted(failed))


Phase-1 checkpoints ready
[11:58:10][_smoke] steps/epoch 50 | total 50 | warmup 3 | lr body 0.001 head 0.001
[11:58:16][_smoke] epoch  1 loss 3.1544 | val strict macro-F1 0.0016 (P 0.0016 R 0.0016) | 0.1 min
[11:58:22][_smoke] VAL best epoch 1: greedy 0.0012 | viterbi 0.0016 | viterbi+propagation['PREC'] 0.0032
[11:58:22][_smoke] VAL per label F1: APP=0.000 RESP=0.000 A.COUNSEL=0.000 R.COUNSEL=0.000 JUDGE=0.000 WIT=0.000 AUTH=0.000 COURT=0.000 STAT=0.000 PREC=0.039 DATE=0.000 CASENO=0.000
[11:58:35][_smoke] TEST: 2099 entities in 35 docs -> /tmp/cs781/p2/smoke/_smoke/submission.zip
[11:58:35][_smoke] DONE
safety test passed in 34s
starting the experiments

round 3 twin: p2_llama_lorma_w1024_all     (val F1 of p2_llama_lorma_w1024: 0.6335, epochs 14)
round 3 twin: p2_llama_lorma_w1024_all_s14 (val F1 of p2_llama_lorma_w1024: 0.6335, epochs 14)
round 3 twin: p2_llama_lorma_all           (val F1 of p2_llama_lorma: 0.6074, epochs 7)
round 3 twin: p2_llama_lora_all            (val F1 of p2_

In [29]:
# ===================== P2 CELL 8 - Results table + submission zips =====================
ZIPS = f"{WORK}/zips"; os.makedirs(ZIPS, exist_ok=True)
rows = []
for n, _ in RUNS:
    p = f"{OUT}/{n}/summary.json"
    if not os.path.exists(p): continue
    s = json.load(open(p)); rows.append(s)
    if os.path.exists(f"{OUT}/{n}/submission.zip"):
        shutil.copy(f"{OUT}/{n}/submission.zip", f"{ZIPS}/{n}.zip")
print(f"{'run':16s} {'model':7s} {'method':7s} {'params':>8s} {'trainable':>10s} {'val F1':>7s} {'P':>6s} {'R':>6s} "
      f"{'greedy':>7s} {'viterbi':>7s} {'ep':>3s} {'train':>6s} {'GPU GB':>6s} {'test s':>6s}")
full_data = [s for s in rows if s.get("train_all")]
rows = [s for s in rows if not s.get("train_all")]
for s in rows:
    m = s["method"] + ("+KD" if s.get("kd_teacher") else "") + ("" if s.get("bidir", 1) else "/causal")
    print(f"{s['run']:16s} {s['arch']:7s} {m:7s} {s['params_total'] / 1e6:7.1f}M {s['params_trainable'] / 1e6:9.2f}M "
          f"{s['val_f1']:7.4f} {s['val_p']:6.3f} {s['val_r']:6.3f} {s['val_f1_greedy']:7.4f} {s['val_f1_viterbi']:7.4f} "
          f"{s['best_epoch']:3d} {s['train_minutes']:5.1f}m {s['peak_gpu_gb']:6.2f} {s['infer_test_seconds']:6.0f}")
print("\nper-label validation F1:")
print(f"{'run':16s} " + " ".join(f"{l[:6]:>6s}" for l in LABELS))
for s in rows:
    print(f"{s['run']:16s} " + " ".join(f"{s['val_per_label'][l][2]:6.3f}" for l in LABELS))
if full_data:
    print("\nfull-data models (trained on all 70 documents, no validation score):",
          ", ".join(f"{s['run']} ({s['best_epoch']} epochs, {s['train_minutes']:.1f}m)" for s in full_data))
json.dump(rows + full_data, open(f"{WORK}/p2_results.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{WORK}/p2_results.json", path_in_repo="phase2/results/p2_results.json",
                repo_id=CKPT_REPO, repo_type="model")
print("\nzips ready in", ZIPS, ":", sorted(os.listdir(ZIPS)))


No files have been modified since last commit. Skipping to prevent empty commit.


run              model   method    params  trainable  val F1      P      R  greedy viterbi  ep  train GPU GB test s
p2_llama_full    llama   full      107.8M    107.78M  0.5378  0.592  0.567  0.3819  0.4967  11   8.5m   6.45     10
p2_gpt2_lorma    gpt2    lorma     112.2M      1.20M  0.2082  0.243  0.223  0.1037  0.1986  15   7.7m   3.49     10
p2_encdec_lora   encdec  lora       67.0M      0.51M  0.3813  0.348  0.433  0.2767  0.3706  16   5.0m   2.24      6
p2_shared_kd     shared  full+KD    58.7M     58.75M  0.5316  0.552  0.547  0.3997  0.4940  12  11.2m   6.33     10
p2_shared_full   shared  full       58.7M     58.75M  0.4952  0.477  0.526  0.3703  0.4744   9   7.8m   5.74     11
p2_llama_lora    llama   lora      108.8M      1.00M  0.5880  0.591  0.617  0.4214  0.5045   7   6.1m   4.26     10
p2_llama_lorma   llama   lorma     108.6M      0.81M  0.6074  0.600  0.651  0.4225  0.5260   7   5.9m   3.96     11
p2_gpt2_full     gpt2    full      111.0M    111.03M  0.2174  0.204  0.2

# Ensemble

In [33]:
# ===================== P2 CELL 9 - Ensemble of the best fine-tuned models (chosen on validation) =====================
if torch.cuda.device_count():
    r = subprocess.run([sys.executable, f"{WORK}/p2ens.py", "--raw_dir", RAW, "--split", f"{P2}/data/split.json",
                        "--tok", TOK_PATH, "--out_dir", OUT, "--ckpt_dir", CK, "--hf_repo", CKPT_REPO],
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True)
    print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "[ensemble]" in l or "Error" in l or "Traceback" in l][-40:]))
    if r.returncode != 0:
        print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    for o in ["ens_val", "ens_all", "ens_top5", "ens_top8", "ens_top5_twins", "ens_top8_twins"]:
        if os.path.exists(f"{OUT}/{o}/submission.zip"):
            shutil.copy(f"{OUT}/{o}/submission.zip", f"{ZIPS}/{o}.zip")
    print("zips:", sorted(os.listdir(ZIPS)))
else:
    print("CPU session: ensemble skipped")


[12:03:08][ensemble] 10 candidate runs (val F1 >= 0.3): p2_llama_lorma_w1024=0.634, p2_llama_lorma=0.607, p2_llama_lora=0.588, p2_shared_kd_lorma=0.572, p2_llama_lorma_bilstm=0.552, p2_llama_full=0.538, p2_llama_causal=0.537, p2_shared_kd=0.532, p2_shared_full=0.495, p2_gpt2_full_causal=0.442
[12:03:08][ensemble] full-data twins: {'p2_llama_lorma_w1024': ['p2_llama_lorma_w1024_all', 'p2_llama_lorma_w1024_all_s14'], 'p2_llama_lorma': ['p2_llama_lorma_all', 'p2_llama_lorma_all_s14'], 'p2_llama_lora': ['p2_llama_lora_all'], 'p2_shared_kd_lorma': ['p2_shared_kd_lorma_all'], 'p2_llama_full': ['p2_llama_full_all'], 'p2_llama_causal': ['p2_llama_causal_all']}
[12:03:26][ensemble] single-model val F1 (with propagation): p2_llama_lorma_w1024=0.6337, p2_llama_lorma=0.6084, p2_llama_lora=0.5882, p2_shared_kd_lorma=0.5687, p2_llama_lorma_bilstm=0.5517, p2_llama_full=0.5371, p2_llama_causal=0.5369, p2_shared_kd=0.5319, p2_shared_full=0.4973, p2_gpt2_full_causal=0.4462
[12:03:40][ensemble]   + p2_ll